# Delta Lake Fundamentals + MERGE

In [0]:
%sql
use catalog workspace;
use schema customer360;

select current_database(), current_catalog(), current_schema();

In [0]:
%sql
select * from customer360.customers;

In [0]:
cust_data = [
    (1001, "John Smith", "Active", "Dallas"),
    (1002, "Mary Jones", "Active", "Plano"),
    (1003, "Steve Brown", "Active", "Frisco")
]

cust_columns = ["custId", "custName", "status", "city"]

dfCustomers = spark.createDataFrame(cust_data, cust_columns)
dfCustomers.write.format("delta").mode("overwrite").saveAsTable("workspace.customer360.customer_master")

In [0]:
%sql
select * from customer360.customer_master;



In [0]:
%sql
DESCRIBE DETAIL customer360.customer_master;


In [0]:
%sql
DESCRIBE HISTORY customer360.customer_master;


In [0]:
customer_updates = [
    (1002, "Mary Jones", "Inactive", "Plano"),
    (1003, "Steve brown", "Active", "McKinney"),
    (1004, "Sarah Wilson", "Active", "Irving")
]

dfUpdates = spark.createDataFrame(customer_updates, cust_columns)

display(dfUpdates)

# DELTA MERGE

````
Lookup target    
     ↓
Exists?
 /       \
Yes       No
 ↓         ↓
UPDATE    INSERT
````
Delta gives us **MERGE**.

Delta MERGE's **WHEN MATCHED UPDATE** and **WHEN NOT MATCHED INSERT** can be implemented in **Informatica** using a Lookup to determine whether the key exists, followed by a **Router** and **Update Strategy** transformations to route records as **DD_UPDATE** or **DD_INSERT**.
````
NEW
ISNULL(LKP_custId)

CHANGED
NOT ISNULL(LKP_custId)
AND (
    status <> LKP_status
 OR city   <> LKP_city
 OR custName <> LKP_custName
)

UNCHANGED
existing row but no attributes changed
````

In [0]:
from delta.tables import DeltaTable

customerMaster = DeltaTable.forName(spark, "customer360.customer_master")
(
customerMaster.alias("target").merge(
    dfUpdates.alias("source"),
    "target.custId = source.custId"
)
.whenMatchedUpdateAll()
.whenNotMatchedInsertAll()
.execute()
)

display(spark.table("customer360.customer_master"))

In [0]:
from delta.tables import DeltaTable

customerMaster = DeltaTable.forName(spark,"customer360.customer_master")
(
    customerMaster.alias("target").merge(
        dfUpdates.alias("source"),
        "target.custId = source.custId"
    )
    .whenMatchedUpdate(
        condition = """ 
                        target.status <> source.status or target.city <> source.city or target.custName <> source.custName
                    """,
        set = {
            "status": "source.status",            
            "city": "source.city",
            "custName": "source.custName"
        } 
    )
    .whenNotMatchedInsertAll()
    .execute()
)

display(spark.table("customer360.customer_master"))


### NOTE:

- **.whenMatchedUpdate()** --> updates only the columns listed in set of the function
    ````
    change the custName in dfUpdates --> run cell 11 --> It will not show the changes as the custName is not listed . Once you add the column in set it will update the custName
    ````

- **.whenMatchedUpdateAll()** --> updates all the columns of target with the value of Source

# DELTA HISTORY

**DECRIBE HISTORY**
- shows the channges made on the table, version 0 being the start.
- tableName@V0 / tablename@V4

In [0]:
%sql
DESCRIBE HISTORY workspace.customer360.customer_master;

In [0]:
%sql
--original table before Insert and Update
SELECT *
FROM workspace.customer360.customer_master
VERSION AS OF 0;

--after insert/Update 
SELECT *
FROM workspace.customer360.customer_master;

# Schema Enforcement
- Delta protects the table from **incompatible schema** changes by default.

In [0]:
bad_data = [
    (1005, "David Lee", "Active", "Dallas", "EXTRA")
]

bad_columns = [
    "custId",
    "custName",
    "status",
    "city",
    "unexpectedColumn"
]

dfBad = spark.createDataFrame(
    bad_data,
    bad_columns
)

dfBad.write.format("delta").mode("overwrite").saveAsTable("workspace.customer360.customer_master")

Adding Column to the customer_master


In [0]:
from pyspark.sql.functions import *

dfCustomers = dfCustomers.withColumn("email", trim(lower(concat(regexp_replace(col("custName"), " ", ""), lit("@gmail.com")))))

dfCustomers.write.format("delta").option("mergeSchema", "true").mode("overwrite").saveAsTable("workspace.customer360.customer_master")

display(spark.table("customer360.customer_master"))

In [0]:
%sql
DESCRIBE workspace.customer360.customer_master;

## Mini Challenge

I'm giving you the requirement, not the implementation.

**Create another incoming feed:**
````
custId   custName       status       city

1001     John Smith     Suspended    Dallas
1003     Steve Brown    Active       McKinney
1005     David Lee      Active       Frisco
````
**Business requirements:**
````
1001 exists → update status.
1003 exists but nothing changed → **don't unnecessarily update it.**
1005 doesn't exist → insert.

**HINT** --> use condition/set in .when MatchedUpdate()

````
**Your task:**

Create:
````
    dfIncrementalCustomers
````
and use **Delta MERGE** to apply those rules to:
````
    workspace.customer360.customer_master
````

In [0]:
cust_updates1 = [
    (1001, "John Smith", "Suspended", "Dallas"),
    (1003, "Steve Brown", "Active", "McKinney"),
    (1005, "David Lee", "Active", "Frisco")
]

cust_columns= ["custId", "custName", "status", "city"]

dfUpdates1 = spark.createDataFrame(cust_updates1, cust_columns)

display(dfUpdates1)

In [0]:
%sql
select * from customer360.customer_master;

In [0]:
from delta.tables import DeltaTable
from pyspark.sql.functions import *

# --Add email to source file, inorder to have same schema structure in target and source
dfUpdates1 = dfUpdates1.withColumn("email", trim(lower(concat(regexp_replace(col("custName"), " ", ""), lit("@gmail.com")))))

customerMaster = DeltaTable.forName(spark,"customer360.customer_master")

(
    customerMaster.alias("target").merge(
        dfUpdates1.alias("source"),
        "target.custId = source.custId"
    )
    .whenMatchedUpdate(
        condition = """ 
                        target.status <> source.status or target.city <> source.city or target.custName <> source.custName
                    """,
        set = {
            "status": "source.status",            
            "city": "source.city",
            "custName": "source.custName"
        } 
    )
    .whenNotMatchedInsertAll()
    .execute()
)

dfIncrementalCustomers = spark.table("customer360.customer_master").orderBy("custId")
display(dfIncrementalCustomers)

In [0]:
%sql
DESCRIBE HISTORY customer360.customer_master;
